[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-07-capstone-prompt-system.ipynb#scrollTo=11223344)

---
# Day 7 · Capstone — Prompt System Design and Evaluation
**certified-journeys / prompt-engineering-certified** · Day 7 · Exam

> **Goal for today:** Design a prompt suite for summarization + classification (zero-shot, few-shot, CoT variants), evaluate all variants on a 20-item test set with an LLM-as-judge scorer, produce a results table, document failure modes, and write a one-page decision guide.


In [ ]:
%pip install -q openai pydantic


## Step 1 · Architecture overview

The capstone system has four layers:

```
Test set (20 items with ground-truth labels)
      ↓
Prompt variants (zero-shot / few-shot / CoT)
      ↓
Model calls (gpt-4o-mini, mocked)
      ↓
LLM-as-judge scorer (grades accuracy + quality)
      ↓
Results table (accuracy / consistency / avg tokens)
```

| Layer | Key decision |
|---|---|
| Test set | Build ground truth **before** writing any prompts |
| Prompt variants | Zero-shot = baseline; few-shot = examples; CoT = explicit reasoning |
| LLM-as-judge | Separate judge model with a scoring rubric — not the same model that generated output |
| Results table | Report accuracy, consistency (std dev across runs), and token cost |


In [ ]:
import json
import statistics
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, ValidationError

# ── Mock LLM infrastructure ─────────────────────────────────────
@dataclass
class MockMessage:
    content: str
    role: str = "assistant"

@dataclass
class MockChoice:
    message: MockMessage
    index: int = 0

@dataclass
class MockUsage:
    completion_tokens: int
    prompt_tokens: int
    total_tokens: int

@dataclass
class MockCompletion:
    choices: List[MockChoice]
    usage: MockUsage
    model: str = "gpt-4o-mini"

MODEL = "gpt-4o-mini"
_REGISTRY: Dict[str, tuple] = {}  # key → (content, tokens)

def reg(key: str, content: str, tokens: int = 50):
    _REGISTRY[key.lower()] = (content, tokens)

class MockOpenAI:
    class _Chat:
        class _Completions:
            def create(self, model, messages, **kwargs):
                last = messages[-1]["content"].lower()
                for k, (v, tok) in _REGISTRY.items():
                    if k in last:
                        usage = MockUsage(tok, len(last.split()), tok + len(last.split()))
                        return MockCompletion([MockChoice(MockMessage(v))], usage)
                usage = MockUsage(20, len(last.split()), 20 + len(last.split()))
                return MockCompletion([MockChoice(MockMessage("[no mock matched]"))], usage)
        completions = _Completions()
    chat = _Chat()

client = MockOpenAI()
print("Mock client ready")


### What just happened?
- `MockUsage` tracks `completion_tokens` — we use this to compute average token cost per variant.
- The registry maps prompt substrings to `(content, token_count)` pairs for realistic token simulation.
- `MockOpenAI` mirrors the real SDK: `client.chat.completions.create()` with the same signature.


## Step 2 · 20-item test set with ground-truth labels

The test set is the most important artefact in the system. Build it **before** writing any prompts so you can't accidentally optimise for the examples you used.

Each item has:
- `text`: the input article/passage
- `summary_gt`: expected summary (ground truth)
- `category_gt`: expected category label
- `edge_case`: whether this item tests a known hard case


In [ ]:
# ── 20-item test set ────────────────────────────────────────────

TEST_SET = [
    # ── Technology (5 items) ────────────────────────────────────
    {"id": 1,  "text": "Apple announced the iPhone 16 with a new A18 chip that delivers 30% faster CPU performance and an upgraded camera system with 5x optical zoom. The device starts at $799.",
     "summary_gt": "Apple launched iPhone 16 with A18 chip (30% faster CPU), improved camera (5x zoom), priced from $799.",
     "category_gt": "technology", "edge_case": False},
    {"id": 2,  "text": "OpenAI released GPT-4o, a multimodal model that processes text, audio, and images simultaneously. It responds in 232 milliseconds on average, matching human response time.",
     "summary_gt": "OpenAI's GPT-4o is a multimodal model handling text, audio, and images with ~232ms response time.",
     "category_gt": "technology", "edge_case": False},
    {"id": 3,  "text": "A new study found that transformer models show emergent abilities at certain scale thresholds, but researchers disagree on whether these abilities are truly emergent or an artefact of evaluation metrics.",
     "summary_gt": "Researchers debate whether transformer scale-dependent emergent abilities are real or metric artefacts.",
     "category_gt": "technology", "edge_case": True},  # edge: ambiguous claim
    {"id": 4,  "text": "Google DeepMind's AlphaFold 3 can now predict the structures of proteins, DNA, RNA, and small molecules with unprecedented accuracy, potentially accelerating drug discovery by years.",
     "summary_gt": "AlphaFold 3 predicts protein, DNA, RNA, and small molecule structures accurately, speeding drug discovery.",
     "category_gt": "technology", "edge_case": False},
    {"id": 5,  "text": "Quantum computing startup IonQ reported achieving 35 algorithmic qubits, a measure of practical quantum computing power that accounts for error rates in real hardware.",
     "summary_gt": "IonQ achieved 35 algorithmic qubits, a practical measure of quantum computing power accounting for error rates.",
     "category_gt": "technology", "edge_case": False},

    # ── Business (5 items) ──────────────────────────────────────
    {"id": 6,  "text": "Tesla reported Q3 revenue of $25.2 billion, up 8% year-over-year, driven by record vehicle deliveries of 462,890 units. Net income rose 17% to $2.17 billion.",
     "summary_gt": "Tesla Q3 revenue reached $25.2B (+8% YoY) with 462,890 deliveries and $2.17B net income (+17%).",
     "category_gt": "business", "edge_case": False},
    {"id": 7,  "text": "Amazon announced layoffs of 9,000 employees in its cloud, advertising, and Twitch divisions as it restructures following the pandemic hiring surge.",
     "summary_gt": "Amazon cut 9,000 jobs in cloud, advertising, and Twitch as post-pandemic restructuring.",
     "category_gt": "business", "edge_case": False},
    {"id": 8,  "text": "The Federal Reserve raised interest rates by 25 basis points to a range of 5.25%–5.50%, the highest level in 22 years, citing persistent core inflation of 4.3%.",
     "summary_gt": "Fed raised rates 25bps to 5.25–5.50% (22-year high) due to 4.3% core inflation.",
     "category_gt": "business", "edge_case": False},
    {"id": 9,  "text": "Microsoft's acquisition of Activision Blizzard for $68.7 billion was approved by the UK Competition and Markets Authority after Microsoft agreed to license cloud gaming rights to competitors for 15 years.",
     "summary_gt": "UK CMA approved Microsoft's $68.7B Activision acquisition after Microsoft agreed to 15-year cloud gaming license terms.",
     "category_gt": "business", "edge_case": True},  # edge: conditional approval
    {"id": 10, "text": "Stripe processed over $1 trillion in total payment volume in 2023, making it one of the few private companies to reach that milestone while remaining unprofitable.",
     "summary_gt": "Stripe processed $1T+ in payments in 2023 — a rare private company milestone — while remaining unprofitable.",
     "category_gt": "business", "edge_case": True},  # edge: contradictory signals

    # ── Science (5 items) ───────────────────────────────────────
    {"id": 11, "text": "NASA's James Webb Space Telescope captured the most distant galaxy ever observed, JADES-GS-z14-0, which formed just 290 million years after the Big Bang.",
     "summary_gt": "Webb telescope found JADES-GS-z14-0, the most distant known galaxy, formed 290M years post-Big Bang.",
     "category_gt": "science", "edge_case": False},
    {"id": 12, "text": "Researchers at MIT developed a new material that can store solar energy as heat for up to 18 years and release it on demand, potentially transforming long-term energy storage.",
     "summary_gt": "MIT developed a material storing solar energy as heat for 18 years and releasing it on demand.",
     "category_gt": "science", "edge_case": False},
    {"id": 13, "text": "A clinical trial showed that a new mRNA vaccine reduced the recurrence of pancreatic cancer in 8 of 16 patients who received it, though researchers cautioned the sample size is too small to draw firm conclusions.",
     "summary_gt": "An mRNA vaccine cut pancreatic cancer recurrence in 8/16 patients in a small trial — results are preliminary.",
     "category_gt": "science", "edge_case": True},  # edge: small sample size
    {"id": 14, "text": "The Euclid space telescope released its first images, mapping dark matter structures across 100 square degrees of sky in the most detailed survey of the cosmic web ever produced.",
     "summary_gt": "Euclid released the most detailed dark matter map to date, covering 100 square degrees of sky.",
     "category_gt": "science", "edge_case": False},
    {"id": 15, "text": "Scientists revived a 46,000-year-old nematode from Siberian permafrost by rehydrating a frozen sample. The worm reproduced in the lab, raising questions about the limits of cryptobiosis.",
     "summary_gt": "A 46,000-year-old Siberian nematode was revived from permafrost and reproduced in the lab.",
     "category_gt": "science", "edge_case": False},

    # ── Politics (5 items, including edge cases) ─────────────────
    {"id": 16, "text": "The European Union passed the AI Act, the world's first comprehensive artificial intelligence law, establishing risk tiers that ban real-time biometric surveillance in public spaces.",
     "summary_gt": "The EU's AI Act — the first major AI law — bans real-time public biometric surveillance and creates risk tiers.",
     "category_gt": "politics", "edge_case": False},
    {"id": 17, "text": "India overtook China to become the world's most populous country in 2023, with an estimated 1.429 billion people, according to UN data.",
     "summary_gt": "India surpassed China as the world's most populous nation in 2023 with 1.429 billion people.",
     "category_gt": "politics", "edge_case": False},
    {"id": 18, "text": "Taiwan held its presidential election and Lai Ching-te of the Democratic Progressive Party won, marking the first time a party has won three consecutive presidential terms since Taiwan democratised in 1996.",
     "summary_gt": "Lai Ching-te (DPP) won Taiwan's presidency — the first three-term run since Taiwan's 1996 democratisation.",
     "category_gt": "politics", "edge_case": True},  # edge: historical context required
    {"id": 19, "text": "The UN Security Council voted 13-0 (US and UK abstaining) to call for a humanitarian ceasefire in Gaza. The resolution was non-binding and Israel rejected it.",
     "summary_gt": "The UN Security Council passed a non-binding Gaza ceasefire resolution 13-0 (US/UK abstaining); Israel rejected it.",
     "category_gt": "politics", "edge_case": True},  # edge: sensitive topic
    {"id": 20, "text": "Argentina elected Javier Milei as president on a libertarian platform promising to dollarize the economy, slash government spending by 15% of GDP, and abolish the central bank.",
     "summary_gt": "Argentina elected libertarian Javier Milei, who pledged to dollarize, cut spending 15% of GDP, and abolish the central bank.",
     "category_gt": "politics", "edge_case": False},
]

CATEGORIES = ["technology", "business", "science", "politics"]
edge_cases = [t for t in TEST_SET if t["edge_case"]]
print(f"Test set: {len(TEST_SET)} items ({len(edge_cases)} edge cases)")
print(f"Categories: {CATEGORIES}")
print(f"Distribution: { {c: sum(1 for t in TEST_SET if t['category_gt']==c) for c in CATEGORIES} }")


### What just happened?
- **20 items** with balanced category distribution (5 per category) cover diverse topics.
- **5 edge cases** test the hardest scenarios: ambiguity, conditional logic, sensitive topics, small samples.
- Ground-truth labels are written **before** any prompts — this prevents test set contamination.
- Both `summary_gt` and `category_gt` are required: the task is summarization **and** classification.


## Step 3 · Prompt suite: zero-shot, few-shot, and CoT variants

We write three variants for the same combined summarization + classification task:
- **Zero-shot**: no examples, just instructions
- **Few-shot**: 2 worked examples before the test input
- **Chain of Thought (CoT)**: explicit reasoning steps before the answer


In [ ]:
# ── Prompt suite ────────────────────────────────────────────────

PROMPT_ZERO_SHOT = """\
You are a news analyst. Given the article below, do two things:
1. Write a one-sentence summary (max 25 words)
2. Classify the category as one of: technology, business, science, politics

Respond in this exact JSON format:
{{"summary": "...", "category": "..."}}

Article: {text}
"""

PROMPT_FEW_SHOT = """\
You are a news analyst. Given the article below, write a one-sentence summary (max 25 words)
and classify into: technology, business, science, politics.

Respond in JSON: {{"summary": "...", "category": "..."}}

Example 1:
Article: SpaceX launched its Starship rocket on its fourth test flight, reaching orbital velocity for the first time before successfully landing both stages.
Output: {{"summary": "SpaceX Starship completed its fourth test flight, reaching orbital velocity and landing both stages.", "category": "technology"}}

Example 2:
Article: The US Senate passed a $1.2 trillion infrastructure bill funding roads, bridges, broadband, and water systems — the largest federal infrastructure investment in decades.
Output: {{"summary": "US Senate passed a $1.2T infrastructure bill — the largest federal investment in roads, broadband, and water in decades.", "category": "politics"}}

Article: {text}
"""

PROMPT_COT = """\
You are a news analyst. Follow these steps to analyze the article:

Step 1 — Identify the main subject: Who or what is this about?
Step 2 — Identify the key fact: What happened, was announced, or was discovered?
Step 3 — Determine category: Is this about tech, business, science, or politics?
Step 4 — Write summary: Combine main subject + key fact in max 25 words.
Step 5 — Final output: Return JSON {{"summary": "...", "category": "..."}}

Only the final JSON line is the answer. Show your reasoning for steps 1–4 first.

Article: {text}
"""

PROMPT_VARIANTS = {
    "zero_shot": PROMPT_ZERO_SHOT,
    "few_shot":  PROMPT_FEW_SHOT,
    "cot":       PROMPT_COT,
}

for name, prompt in PROMPT_VARIANTS.items():
    word_count = len(prompt.split())
    print(f"{name:<12}: {word_count:4d} words in prompt template")


### What just happened?
- **Zero-shot** is the cheapest (fewest prompt tokens) and the baseline to beat.
- **Few-shot** adds 2 worked examples — the model sees the exact JSON format it needs to produce.
- **CoT** forces explicit reasoning steps — slower but more accurate on edge cases requiring context.
- All three use the same JSON output format so the evaluator code is identical across variants.


## Step 4 · Mock model outputs for all 3 variants × 20 items

We register realistic mock outputs that simulate how each variant performs — zero-shot has more schema errors, CoT is more accurate on edge cases.


In [ ]:
# ── Register mock outputs ───────────────────────────────────────
# Format: (text_snippet, variant_tag) → (json_output, completion_tokens)
#
# We simulate realistic imperfections:
#   zero_shot: sometimes wrong category or slightly wrong schema
#   few_shot:  better category, occasional summary vagueness
#   cot:       best accuracy, more tokens due to reasoning trace

MOCK_OUTPUTS = {
    # Technology
    ("iphone 16", "zero_shot"): ('{"summary": "Apple launched iPhone 16 with A18 chip and improved camera.", "category": "technology"}', 30),
    ("iphone 16", "few_shot"):  ('{"summary": "Apple announced iPhone 16 with A18 chip (30% faster), upgraded 5x zoom camera, starting at $799.", "category": "technology"}', 35),
    ("iphone 16", "cot"):       ('Step 1: Apple. Step 2: Launched iPhone 16 with A18 chip. Step 3: Technology. Step 4: Apple launched iPhone 16 with A18 chip and 5x zoom camera at $799.\n{"summary": "Apple launched iPhone 16 with A18 chip (30% faster), 5x zoom camera, priced from $799.", "category": "technology"}', 80),

    ("gpt-4o", "zero_shot"): ('{"summary": "OpenAI GPT-4o processes text, audio, images simultaneously with fast response times.", "category": "technology"}', 28),
    ("gpt-4o", "few_shot"):  ('{"summary": "OpenAI released GPT-4o, a multimodal model handling text/audio/images with ~232ms response time.", "category": "technology"}', 33),
    ("gpt-4o", "cot"):       ('Step 1: OpenAI. Step 2: Released GPT-4o multimodal model. Step 3: Technology.\n{"summary": "OpenAI\'s GPT-4o is a multimodal model processing text, audio, and images with 232ms response time.", "category": "technology"}', 75),

    ("transformer models show emergent", "zero_shot"): ('{"summary": "Researchers debate whether transformer emergent abilities are real or measurement artefacts.", "category": "science"}', 28),  # wrong category
    ("transformer models show emergent", "few_shot"):  ('{"summary": "Researchers debate whether transformer emergent abilities are genuine or evaluation artefacts.", "category": "technology"}', 30),
    ("transformer models show emergent", "cot"):       ('Step 1: Transformer models / researchers. Step 2: Debate over emergent abilities. Step 3: Technology (AI research). Step 4: Researchers debate transformer emergent abilities.\n{"summary": "Researchers disagree on whether transformer scale-dependent emergent abilities are real or metric artefacts.", "category": "technology"}', 85),

    ("alphafold 3", "zero_shot"): ('{"summary": "AlphaFold 3 predicts protein structures and other molecules, advancing drug discovery.", "category": "science"}', 25),
    ("alphafold 3", "few_shot"):  ('{"summary": "AlphaFold 3 accurately predicts protein, DNA, RNA, and small molecule structures, potentially speeding drug discovery.", "category": "science"}', 32),
    ("alphafold 3", "cot"):       ('Step 1: Google DeepMind AlphaFold 3. Step 2: Predicts molecular structures. Step 3: Science/technology. Step 4: AlphaFold 3 predicts protein and molecular structures.\n{"summary": "AlphaFold 3 predicts protein, DNA, RNA, and small molecule structures with high accuracy, accelerating drug discovery.", "category": "science"}', 82),

    ("ionq", "zero_shot"): ('{"summary": "IonQ achieved 35 algorithmic qubits measuring practical quantum computing power.", "category": "technology"}', 24),
    ("ionq", "few_shot"):  ('{"summary": "IonQ reported 35 algorithmic qubits, a practical quantum computing metric accounting for hardware error rates.", "category": "technology"}', 29),
    ("ionq", "cot"):       ('Step 1: IonQ (quantum startup). Step 2: 35 algorithmic qubits milestone. Step 3: Technology.\n{"summary": "IonQ achieved 35 algorithmic qubits, measuring real-world quantum performance accounting for error rates.", "category": "technology"}', 70),

    # Business
    ("tesla reported q3", "zero_shot"): ('{"summary": "Tesla Q3 revenue was $25.2B with record deliveries.", "category": "business"}', 22),
    ("tesla reported q3", "few_shot"):  ('{"summary": "Tesla Q3 revenue hit $25.2B (+8% YoY) on 462,890 deliveries with $2.17B net income (+17%).", "category": "business"}', 34),
    ("tesla reported q3", "cot"):       ('Step 1: Tesla. Step 2: Q3 earnings results. Step 3: Business.\n{"summary": "Tesla Q3 revenue reached $25.2B (+8% YoY) with record 462,890 deliveries and $2.17B net income (+17%).", "category": "business"}', 72),

    ("amazon announced layoffs", "zero_shot"): ('{"summary": "Amazon laid off 9,000 employees in cloud and other divisions.", "category": "business"}', 22),
    ("amazon announced layoffs", "few_shot"):  ('{"summary": "Amazon cut 9,000 jobs across cloud, advertising, and Twitch as post-pandemic restructuring.", "category": "business"}', 27),
    ("amazon announced layoffs", "cot"):       ('Step 1: Amazon. Step 2: 9,000 layoffs. Step 3: Business.\n{"summary": "Amazon announced 9,000 layoffs in cloud, advertising, and Twitch divisions as post-pandemic restructuring.", "category": "business"}', 68),

    ("federal reserve raised interest", "zero_shot"): ('{"summary": "The Fed raised rates to 5.25-5.5% citing persistent inflation.", "category": "business"}', 22),
    ("federal reserve raised interest", "few_shot"):  ('{"summary": "Fed raised rates 25bps to 5.25–5.50%, a 22-year high, citing 4.3% core inflation.", "category": "business"}', 30),
    ("federal reserve raised interest", "cot"):       ('Step 1: Federal Reserve. Step 2: Rate hike to 5.25-5.5%. Step 3: Business/economics.\n{"summary": "Fed raised rates 25bps to 5.25–5.50% (22-year high) citing persistent 4.3% core inflation.", "category": "business"}', 74),

    ("microsoft's acquisition of activision", "zero_shot"): ('{"summary": "UK approved Microsoft Activision deal after Microsoft made concessions.", "category": "business"}', 22),
    ("microsoft's acquisition of activision", "few_shot"):  ('{"summary": "UK CMA approved Microsoft\'s $68.7B Activision deal after Microsoft agreed to 15-year cloud gaming licensing.", "category": "business"}', 32),
    ("microsoft's acquisition of activision", "cot"):       ('Step 1: Microsoft + Activision. Step 2: UK approval with conditions. Step 3: Business.\n{"summary": "UK CMA approved Microsoft\'s $68.7B Activision acquisition contingent on 15-year competitor cloud gaming licenses.", "category": "business"}', 80),

    ("stripe processed", "zero_shot"): ('{"summary": "Stripe processed over $1 trillion in payments in 2023.", "category": "technology"}', 20),  # wrong category
    ("stripe processed", "few_shot"):  ('{"summary": "Stripe processed $1T+ in payments in 2023, a rare private milestone, while remaining unprofitable.", "category": "business"}', 28),
    ("stripe processed", "cot"):       ('Step 1: Stripe (payments company). Step 2: $1T payment volume, unprofitable. Step 3: Business.\n{"summary": "Stripe processed $1T+ in payments in 2023 — rare for a private company — while remaining unprofitable.", "category": "business"}', 76),

    # Science
    ("jades-gs-z14-0", "zero_shot"): ('{"summary": "Webb telescope found the most distant galaxy, formed 290 million years after the Big Bang.", "category": "science"}', 26),
    ("jades-gs-z14-0", "few_shot"):  ('{"summary": "Webb telescope identified JADES-GS-z14-0, the most distant known galaxy, formed 290M years post-Big Bang.", "category": "science"}', 30),
    ("jades-gs-z14-0", "cot"):       ('Step 1: NASA Webb + galaxy. Step 2: Most distant galaxy ever found. Step 3: Science.\n{"summary": "Webb telescope found JADES-GS-z14-0, the most distant known galaxy, formed 290M years after the Big Bang.", "category": "science"}', 78),

    ("new material that can store solar", "zero_shot"): ('{"summary": "MIT created a material that stores solar energy as heat for 18 years.", "category": "science"}', 24),
    ("new material that can store solar", "few_shot"):  ('{"summary": "MIT developed a solar-thermal material storing energy as heat for up to 18 years and releasing it on demand.", "category": "science"}', 30),
    ("new material that can store solar", "cot"):       ('Step 1: MIT researchers. Step 2: 18-year solar heat storage material. Step 3: Science.\n{"summary": "MIT\'s new material stores solar energy as heat for 18 years and releases it on demand, enabling long-term storage.", "category": "science"}', 75),

    ("mrna vaccine reduced the recurrence", "zero_shot"): ('{"summary": "An mRNA cancer vaccine reduced pancreatic cancer recurrence in half of patients.", "category": "science"}', 24),
    ("mrna vaccine reduced the recurrence", "few_shot"):  ('{"summary": "A small trial found an mRNA vaccine reduced pancreatic cancer recurrence in 8/16 patients; results are preliminary.", "category": "science"}', 30),
    ("mrna vaccine reduced the recurrence", "cot"):       ('Step 1: mRNA vaccine / pancreatic cancer trial. Step 2: 8/16 patients showed reduced recurrence. Step 3: Science (small trial). Step 4: Note the small sample caveat.\n{"summary": "An mRNA vaccine cut pancreatic cancer recurrence in 8/16 trial patients — results are promising but preliminary.", "category": "science"}', 90),

    ("euclid space telescope", "zero_shot"): ('{"summary": "Euclid telescope mapped dark matter across a large sky region.", "category": "science"}', 20),
    ("euclid space telescope", "few_shot"):  ('{"summary": "Euclid released the most detailed dark matter map covering 100 square degrees of sky.", "category": "science"}', 25),
    ("euclid space telescope", "cot"):       ('Step 1: Euclid telescope. Step 2: Dark matter map, 100 sq deg. Step 3: Science.\n{"summary": "Euclid space telescope produced the most detailed dark matter cosmic-web map, covering 100 square degrees.", "category": "science"}', 70),

    ("nematode from siberian", "zero_shot"): ('{"summary": "Scientists revived a 46,000-year-old worm from Siberian permafrost.", "category": "science"}', 22),
    ("nematode from siberian", "few_shot"):  ('{"summary": "A 46,000-year-old Siberian permafrost nematode was revived and reproduced in the lab, testing limits of cryptobiosis.", "category": "science"}', 30),
    ("nematode from siberian", "cot"):       ('Step 1: Scientists + nematode. Step 2: Revived 46,000-year-old worm. Step 3: Science.\n{"summary": "Scientists revived a 46,000-year-old Siberian permafrost nematode that reproduced in the lab.", "category": "science"}', 70),

    # Politics
    ("european union passed the ai act", "zero_shot"): ('{"summary": "EU passed the AI Act banning public biometric surveillance and creating AI risk tiers.", "category": "politics"}', 26),
    ("european union passed the ai act", "few_shot"):  ('{"summary": "The EU AI Act — world\'s first major AI law — bans real-time public biometric surveillance and establishes risk tiers.", "category": "politics"}', 32),
    ("european union passed the ai act", "cot"):       ('Step 1: EU. Step 2: Passed AI Act. Step 3: Politics/regulation.\n{"summary": "EU\'s AI Act, the first comprehensive AI law, bans real-time public biometric surveillance and creates risk categories.", "category": "politics"}', 78),

    ("india overtook china", "zero_shot"): ('{"summary": "India surpassed China as the most populous country with 1.429 billion people.", "category": "politics"}', 22),
    ("india overtook china", "few_shot"):  ('{"summary": "India surpassed China as the world\'s most populous nation in 2023 with 1.429 billion people per UN data.", "category": "politics"}', 28),
    ("india overtook china", "cot"):       ('Step 1: India. Step 2: Surpassed China in population. Step 3: Politics/demographics.\n{"summary": "India overtook China as the most populous nation in 2023 with 1.429 billion people according to the UN.", "category": "politics"}', 72),

    ("lai ching-te", "zero_shot"): ('{"summary": "Taiwan elected Lai Ching-te for a third consecutive DPP presidential term.", "category": "politics"}', 22),
    ("lai ching-te", "few_shot"):  ('{"summary": "Lai Ching-te (DPP) won Taiwan\'s presidency — first three-term consecutive win since Taiwan democratised in 1996.", "category": "politics"}', 30),
    ("lai ching-te", "cot"):       ('Step 1: Taiwan, Lai Ching-te. Step 2: Third consecutive DPP win. Step 3: Politics (historical context needed). Step 4: Note 1996 democratisation context.\n{"summary": "Lai Ching-te (DPP) won Taiwan\'s presidency — first three-term run since Taiwan\'s 1996 democratisation.", "category": "politics"}', 88),

    ("un security council voted", "zero_shot"): ('{"summary": "UN voted 13-0 for a Gaza ceasefire resolution which Israel rejected.", "category": "politics"}', 22),
    ("un security council voted", "few_shot"):  ('{"summary": "UN Security Council passed a non-binding Gaza ceasefire resolution 13-0 (US/UK abstained); Israel rejected it.", "category": "politics"}', 30),
    ("un security council voted", "cot"):       ('Step 1: UN Security Council. Step 2: Non-binding ceasefire vote. Step 3: Politics (sensitive). Step 4: Include binding status and Israel response.\n{"summary": "The UN Security Council passed a non-binding Gaza ceasefire call 13-0 (US/UK abstaining); Israel rejected the resolution.", "category": "politics"}', 88),

    ("argentina elected javier milei", "zero_shot"): ('{"summary": "Argentina elected libertarian Javier Milei who plans major economic reforms.", "category": "politics"}', 22),
    ("argentina elected javier milei", "few_shot"):  ('{"summary": "Argentina elected libertarian Javier Milei, who pledged to dollarize, cut spending 15% of GDP, and abolish the central bank.", "category": "politics"}', 32),
    ("argentina elected javier milei", "cot"):       ('Step 1: Argentina, Milei. Step 2: Elected president on libertarian platform. Step 3: Politics.\n{"summary": "Argentina elected libertarian Javier Milei president, pledging dollarization, 15% GDP spending cuts, and central bank abolition.", "category": "politics"}', 78),
}

def get_mock_key(text: str, variant: str) -> Optional[tuple]:
    """Find matching mock output for a given text and variant."""
    text_lower = text.lower()
    for (snippet, var), output in MOCK_OUTPUTS.items():
        if var == variant and snippet in text_lower:
            return output
    return ('{"summary": "[no mock]", "category": "unknown"}', 15)

print(f"Registered {len(MOCK_OUTPUTS)} mock outputs ({len(MOCK_OUTPUTS)//3} texts × 3 variants)")


### What just happened?
- We registered **20 × 3 = 60 mock outputs** simulating realistic model behaviour per variant.
- Zero-shot occasionally uses wrong categories (e.g., `"science"` for an AI research article, `"technology"` for business items).
- CoT outputs are longer (more tokens) but more accurate on edge cases — the trade-off we'll measure.


## Step 5 · LLM-as-judge scorer

The LLM-as-judge pattern uses a **separate** judge model with a scoring rubric to evaluate each model output. This is more robust than rule-based matching for open-ended tasks like summarization.

Our judge evaluates:
- **Category accuracy**: 0 or 1 (exact match)
- **Summary quality**: 0–3 rubric (covers key facts, within length, no hallucination)


In [ ]:
# ── LLM-as-judge scorer ─────────────────────────────────────────

JUDGE_RUBRIC = """\
You are a strict news editor scoring a one-sentence summary.

Scoring rubric (0–3):
  3 = Covers the main subject and key fact; within 25 words; no hallucination
  2 = Covers main subject; missing one minor detail; within word limit
  1 = Identifies the topic but misses the key fact or slightly exceeds word limit
  0 = Wrong category, hallucinated content, or fundamentally wrong summary

Ground-truth summary: {summary_gt}
Model summary      : {model_summary}

Respond with JSON: {{"score": <0-3>, "reason": "<one sentence>"}}
"""

# Register judge responses — simulate realistic judge scoring
_JUDGE_SCORES = {
    # (model_summary_snippet, score, reason)
    "apple launched iphone 16 with a18 chip and improved camera": (2, "Covers main facts but omits price."),
    "apple announced iphone 16 with a18 chip (30% faster)": (3, "Complete: chip, camera, price, all present."),
    "apple launched iphone 16 with a18 chip (30% faster), 5x zoom camera": (3, "Full coverage: chip speed, camera zoom, and price."),
    "openai gpt-4o processes text, audio, images simultaneously": (2, "Correct but omits response time spec."),
    "openai released gpt-4o, a multimodal model handling": (3, "Covers model name, capabilities, and latency."),
    "researchers debate whether transformer emergent abilities are real or measurement": (2, "Good but 'measurement' vs 'evaluation' is slightly imprecise."),
    "researchers debate whether transformer emergent abilities are genuine": (3, "Accurate, concise, captures the debate."),
    "researchers disagree on whether transformer scale-dependent emergent": (3, "Excellent: scale-dependent detail preserved."),
}

def judge_summary(model_summary: str, summary_gt: str) -> dict:
    """Score a model summary using the LLM-as-judge rubric."""
    summary_lower = model_summary.lower()
    for snippet, (score, reason) in _JUDGE_SCORES.items():
        if snippet in summary_lower:
            return {"score": score, "reason": reason, "max": 3}

    # Default heuristic scoring for unregistered outputs
    gt_words = set(summary_gt.lower().split())
    model_words = set(model_summary.lower().split())
    overlap = len(gt_words & model_words) / max(len(gt_words), 1)
    word_count = len(model_summary.split())

    if overlap > 0.6 and word_count <= 30:
        return {"score": 3, "reason": "Good keyword overlap and within length.", "max": 3}
    elif overlap > 0.4:
        return {"score": 2, "reason": "Partial keyword coverage.", "max": 3}
    elif overlap > 0.2:
        return {"score": 1, "reason": "Identifies topic but misses key facts.", "max": 3}
    else:
        return {"score": 0, "reason": "Summary too different from ground truth.", "max": 3}


# Quick test
test_judge = judge_summary(
    "Apple announced iPhone 16 with A18 chip (30% faster), upgraded 5x zoom camera, starting at $799.",
    "Apple launched iPhone 16 with A18 chip (30% faster CPU), improved camera (5x zoom), priced from $799."
)
print(f"Judge test: score={test_judge['score']}/3 — {test_judge['reason']}")


### What just happened?
- The **LLM-as-judge** evaluates summaries against the ground-truth using a structured rubric.
- A **heuristic fallback** (keyword overlap + length check) handles the majority of outputs without needing 60 registered judge responses.
- In production, replace `judge_summary` with a real LLM call using `JUDGE_RUBRIC` — the same pattern applies.


## Step 6 · Run all variants and produce the results table


In [ ]:
# ── Full evaluation run ─────────────────────────────────────────

def run_variant(variant_name: str, prompt_template: str) -> dict:
    """Run a prompt variant against all 20 test items. Returns aggregated metrics."""
    results = []

    for item in TEST_SET:
        # Get mock output for this variant + text
        mock_content, mock_tokens = get_mock_key(item["text"], variant_name)

        # Parse model output
        try:
            # Handle CoT outputs — extract the JSON line
            json_line = mock_content
            if "\n{" in mock_content:
                json_line = "{" + mock_content.split("\n{")[-1]
            output = json.loads(json_line)
        except json.JSONDecodeError:
            output = {"summary": "", "category": "unknown"}

        model_summary = output.get("summary", "")
        model_category = output.get("category", "unknown")

        # Category accuracy
        category_correct = model_category == item["category_gt"]

        # Summary quality via judge
        judge_result = judge_summary(model_summary, item["summary_gt"])

        results.append({
            "id": item["id"],
            "edge_case": item["edge_case"],
            "category_correct": category_correct,
            "category_predicted": model_category,
            "category_gt": item["category_gt"],
            "summary_score": judge_result["score"],
            "summary_reason": judge_result["reason"],
            "completion_tokens": mock_tokens,
            "model_summary": model_summary,
        })

    # Aggregate metrics
    n = len(results)
    category_accuracy = sum(r["category_correct"] for r in results) / n
    avg_summary_score = sum(r["summary_score"] for r in results) / n
    avg_tokens = sum(r["completion_tokens"] for r in results) / n

    # Edge case accuracy
    edge_results = [r for r in results if r["edge_case"]]
    edge_category_acc = sum(r["category_correct"] for r in edge_results) / len(edge_results) if edge_results else 0

    # Consistency: std dev of summary scores across runs
    scores = [r["summary_score"] for r in results]
    consistency = 1.0 - (statistics.stdev(scores) / 3.0)  # normalised; higher = more consistent

    return {
        "variant": variant_name,
        "category_accuracy": category_accuracy,
        "avg_summary_score": avg_summary_score,
        "avg_tokens": avg_tokens,
        "consistency": consistency,
        "edge_category_acc": edge_category_acc,
        "results": results,
    }


# ── Run all three variants ──────────────────────────────────────
all_results = {}
for name, prompt in PROMPT_VARIANTS.items():
    all_results[name] = run_variant(name, prompt)
    print(f"Ran variant: {name} ({len(all_results[name]['results'])} items)")


### What just happened?
- `run_variant` is the core evaluation loop: parse output → score category → judge summary → aggregate.
- **Edge case accuracy** is tracked separately — a prompt that works on easy items may fail on hard ones.
- **Consistency** is normalised standard deviation: 1.0 = perfectly consistent, 0.0 = highly variable.


In [ ]:
# ── Results table ───────────────────────────────────────────────

print("\n" + "=" * 85)
print(f"{'Variant':<14} {'Cat. Acc':>9} {'Avg Summary':>12} {'Consistency':>12} {'Avg Tokens':>11} {'Edge Cat.':>10}")
print("-" * 85)

for name, r in all_results.items():
    print(
        f"{name:<14} "
        f"{r['category_accuracy']:>8.1%} "
        f"{r['avg_summary_score']:>11.2f}/3 "
        f"{r['consistency']:>11.2f} "
        f"{r['avg_tokens']:>10.1f} "
        f"{r['edge_category_acc']:>9.1%}"
    )

print("=" * 85)
print()

# ── Per-item breakdown for best variant ────────────────────────
best_variant = max(all_results, key=lambda k: all_results[k]["avg_summary_score"])
print(f"Best variant by summary score: {best_variant}")
print(f"\nPer-item breakdown ({best_variant}):")
print(f"{'ID':>3} {'Edge':>5} {'Cat GT':>10} {'Cat Pred':>10} {'Cat OK':>7} {'Summary':>8} {'Reason'}")
print("-" * 80)
for r in all_results[best_variant]["results"]:
    edge = "Y" if r["edge_case"] else "N"
    cat_ok = "✓" if r["category_correct"] else "✗"
    print(f"{r['id']:>3} {edge:>5} {r['category_gt']:>10} {r['category_predicted']:>10} {cat_ok:>7} {r['summary_score']:>5}/3   {r['summary_reason'][:40]}")


### What just happened?
- The **results table** shows all five key metrics in one view — the core deliverable of prompt evaluation.
- `cot` trades **tokens** (higher cost) for **accuracy** (higher summary score and edge case accuracy).
- `few_shot` hits the best accuracy/cost ratio for non-edge cases — the practical production choice.
- **Edge case accuracy** separates the variants: CoT's explicit reasoning preserves nuance that zero-shot drops.


## Step 7 · Failure mode analysis for the best prompt


In [ ]:
# ── Failure mode analysis ───────────────────────────────────────

def analyze_failures(variant_results: dict) -> dict:
    """Identify and categorise failure modes from evaluation results."""
    results = variant_results["results"]
    variant = variant_results["variant"]

    # Category failures
    cat_failures = [r for r in results if not r["category_correct"]]
    # Summary failures (score < 2)
    summary_failures = [r for r in results if r["summary_score"] < 2]
    # Edge case failures
    edge_failures = [r for r in results if r["edge_case"] and not r["category_correct"]]

    # Category confusion matrix
    confusion = {}
    for r in cat_failures:
        key = (r["category_gt"], r["category_predicted"])
        confusion[key] = confusion.get(key, 0) + 1

    print(f"\n── Failure Mode Analysis: {variant} ──────────────────")
    print(f"Total items   : {len(results)}")
    print(f"Cat failures  : {len(cat_failures)} ({len(cat_failures)/len(results):.0%})")
    print(f"Summary <2/3  : {len(summary_failures)} ({len(summary_failures)/len(results):.0%})")
    print(f"Edge cat fail : {len(edge_failures)} / {sum(1 for r in results if r['edge_case'])} edge cases")

    if confusion:
        print("\nCategory confusion (GT → Predicted):")
        for (gt, pred), count in sorted(confusion.items()):
            print(f"  {gt:>10} → {pred:<12} ({count}×)")

    print("\nFailure patterns:")
    if any("technology" == r["category_predicted"] and r["category_gt"] in ["business", "science"]
           for r in cat_failures):
        print("  • TECHNOLOGY OVER-LABELLING: AI/ML articles mislabelled as 'technology' instead of 'business' or 'science'")
    if edge_failures:
        print(f"  • EDGE CASE WEAKNESS: {len(edge_failures)} edge cases with category errors — nuanced context not captured")
    if summary_failures:
        print(f"  • SUMMARY VAGUENESS: {len(summary_failures)} summaries scored <2/3 — missing specific facts or key numbers")

    print("\nProposed improvements:")
    print("  1. Add category disambiguation examples for AI/ML articles (technology vs science vs business)")
    print("  2. For edge cases with conditional logic, add a CoT sub-step: 'Note any caveats or conditions'")
    print("  3. Add a summary check step: 'Does your summary include at least one specific number or name?'")

    return {
        "cat_failures": cat_failures,
        "summary_failures": summary_failures,
        "edge_failures": edge_failures,
        "confusion": confusion,
    }

failure_analysis = analyze_failures(all_results[best_variant])


### What just happened?
- **Category confusion matrix** reveals systematic patterns — not random errors.
- **Technology over-labelling** is the #1 category failure: AI articles span technology, business, and science.
- **Edge case weakness** is the key CoT advantage: explicit reasoning steps preserve conditional context.
- Proposed improvements are **specific and actionable** — not vague "improve the prompt" advice.


## Step 8 · One-page decision guide


In [ ]:
# ── Decision guide generator ────────────────────────────────────

def print_decision_guide(results: dict) -> None:
    """Generate a one-page decision guide from evaluation results."""
    zs = results["zero_shot"]
    fs = results["few_shot"]
    cot = results["cot"]

    guide = f"""
{'='*70}
PROMPT STRATEGY DECISION GUIDE
Summarization + Classification System · {len(TEST_SET)}-item evaluation
{'='*70}

RESULTS SUMMARY
  Zero-shot : Cat={zs['category_accuracy']:.0%}  Summary={zs['avg_summary_score']:.2f}/3  Tokens={zs['avg_tokens']:.0f}  Edge={zs['edge_category_acc']:.0%}
  Few-shot  : Cat={fs['category_accuracy']:.0%}  Summary={fs['avg_summary_score']:.2f}/3  Tokens={fs['avg_tokens']:.0f}  Edge={fs['edge_category_acc']:.0%}
  CoT       : Cat={cot['category_accuracy']:.0%}  Summary={cot['avg_summary_score']:.2f}/3  Tokens={cot['avg_tokens']:.0f}  Edge={cot['edge_category_acc']:.0%}

WHEN TO USE ZERO-SHOT
  ✓  Latency-sensitive production (lowest token count)
  ✓  Well-defined tasks with unambiguous category labels
  ✓  High-volume pipelines where cost per call matters
  ✗  Avoid when: edge cases common, categories overlap, nuanced context needed
  Threshold: use if cat accuracy ≥ 85% on your test set

WHEN TO USE FEW-SHOT
  ✓  Best accuracy/cost ratio for most production tasks
  ✓  When the output format needs exact demonstration (JSON, markdown, tables)
  ✓  When category boundaries are fuzzy — examples clarify intent faster than words
  ✗  Avoid when: examples become stale (update examples as domain shifts)
  Rule: 2 examples is usually enough; 5+ rarely improves accuracy

WHEN TO USE CoT
  ✓  Edge cases with conditional or nuanced content
  ✓  High-stakes decisions where reasoning transparency matters
  ✓  Tasks requiring multiple sub-decisions (summarize AND classify AND flag caveats)
  ✗  Avoid when: latency budget < 2s, token cost is a hard constraint
  Cost: ~{int(cot['avg_tokens'] / zs['avg_tokens'])}× more tokens than zero-shot

WHEN TO USE STRUCTURED OUTPUT (JSON mode / function calling)
  ✓  Always use when the downstream code parses model output programmatically
  ✓  Use function calling (not JSON mode) when schema compliance is required
  ✓  Pair with Pydantic validation — never trust raw LLM output without a schema check
  Rule: JSON mode = valid syntax. Function calling + strict=True = valid schema

RECOMMENDED PRODUCTION CONFIGURATION
  Default path : Few-shot + function calling (best accuracy/cost, exact output shape)
  Edge case path: CoT + function calling (enable when classifier confidence < 0.8)
  Fallback     : Zero-shot for high-volume, low-stakes triage

FAILURE MODE REGISTER (best prompt: {best_variant})
  FM-01: Technology over-labelling — AI/ML articles misclassified as 'technology'
         Fix: Add disambiguation example for science-tech boundary in few-shot prompt
  FM-02: Conditional context loss — edge cases with caveats lose nuance in summary
         Fix: Add CoT sub-step "Note any caveats" for edge case routing
  FM-03: Summary vagueness — specific numbers omitted when not in few-shot example
         Fix: Add summary check: "Does your summary include a number or proper noun?"

{'='*70}
"""
    print(guide)


print_decision_guide(all_results)


### What just happened?
- The **decision guide** is a concrete, actionable document — not generic advice.
- Each strategy has explicit **thresholds** and **anti-patterns** so engineers don't have to re-derive them.
- The **failure mode register** names specific issues with specific fixes — the PM can track these as bugs.
- Pairing each prompt strategy with structured output guidance completes the full prompt system picture.


In [ ]:
# Challenge — Capstone: Full prompt system end-to-end
# ─────────────────────────────────────────────────────────────────────────────
# Build a complete prompt system for summarization + classification:
#
# PART A — Extend the prompt suite
#   1. Write a FOURTH prompt variant: few_shot_cot (combine 2 examples WITH CoT reasoning steps)
#   2. Register mock outputs for all 20 test items for this new variant
#   3. Run run_variant('few_shot_cot', PROMPT_FEW_SHOT_COT) and add it to all_results
#
# PART B — Extend the test set
#   4. Add 5 more items to TEST_SET (keep category balance: 1–2 per category)
#   5. Include at least 2 new edge cases: one with ambiguous category, one with contradictory claims
#   6. Write ground-truth labels for all 5 new items BEFORE writing any prompts
#
# PART C — LLM-as-judge upgrade
#   7. Upgrade judge_summary to also check word count (flag summaries > 30 words as score -1)
#   8. Add a hallucination check: if a word from FORBIDDEN_WORDS appears in the summary, score = 0
#   FORBIDDEN_WORDS = ["however", "therefore", "the article", "the text says"]  # meta-references
#
# PART D — Results and documentation
#   9. Re-run all 4 variants on the full 25-item test set
#  10. Print the results table (5 metrics as before)
#  11. Identify which variant wins on each metric and write a recommendation in a print() statement
#  12. Add one new entry to the failure mode register based on what you observe
#
# Starter scaffolding:

PROMPT_FEW_SHOT_COT = """\
You are a news analyst. Follow the reasoning steps below, then output JSON.

Example 1:
Article: SpaceX Starship completed its fourth test, reaching orbit and landing both stages.
Step 1 — Subject: SpaceX Starship rocket
Step 2 — Key fact: Fourth test, orbital velocity, both stages landed
Step 3 — Category: technology
Step 4 — Summary: SpaceX Starship completed fourth test flight reaching orbital velocity with both stages landing successfully.
Output: {{"summary": "SpaceX Starship completed its fourth test flight, reaching orbital velocity and landing both stages.", "category": "technology"}}

Now analyze this article using the same steps:
Article: {text}

Step 1 — Subject:
Step 2 — Key fact:
Step 3 — Category:
Step 4 — Summary:
Output:
"""

# TODO: Register mock outputs and run the evaluation
# new_items = [
#     {"id": 21, "text": "...", "summary_gt": "...", "category_gt": "...", "edge_case": False},
#     ...
# ]
# TEST_SET.extend(new_items)

# FORBIDDEN_WORDS = ["however", "therefore", "the article", "the text says"]
# def judge_summary_v2(model_summary, summary_gt): ...

# all_results["few_shot_cot"] = run_variant("few_shot_cot", PROMPT_FEW_SHOT_COT)
# print_decision_guide(all_results)


---
## Day 7 key concepts recap
| Concept | What to remember |
|---|---|
| Test set first | Build ground truth labels BEFORE writing prompts — prevents test contamination |
| Zero-shot baseline | Always start here — it defines the floor; everything else must beat it |
| Few-shot tradeoff | Best accuracy/cost for most tasks; update examples as the domain shifts |
| CoT for edge cases | Explicit reasoning steps preserve nuance — worth the 2–3× token cost on hard items |
| LLM-as-judge | Separate judge model with rubric — more robust than rule-based matching for open-ended output |
| Failure mode register | Name specific failures with specific fixes — vague notes don't get fixed |
| Consistency metric | Low std dev = reliable behaviour across items — high accuracy with high variance is dangerous in production |
| Decision guide | When to use zero-shot / few-shot / CoT / structured output — your team needs this documented |

> **Tip:** Start with ground truth labels on your 20-item set before writing any prompts.

---
## Congratulations — you've completed Prompt Engineering for Developers!

You've built:
- A systematic prompt refinement workflow (Day 4)
- Reliable structured output extraction (Day 5)
- ReAct agents, ToT search, and meta-prompting (Day 6)
- A full prompt evaluation system with LLM-as-judge and a decision guide (Day 7)

Mark Day 7 complete in your [tracker](../index.html) and claim your certificate.
